# 10. Tables for the Thesis

Builds the tables that document the time-window table T1g, the four scenarios and the CRI results, from
the pipeline outputs only (notebooks 01, 06 and 08) and the input files. It does not use the sensitivity
analysis of notebook 09.

The comparison with the article reads the published results from git commit `7cef902`, so it stays
reproducible after the data files are regenerated.

Outputs in `../docs/thesis/`: one CSV per table (raw values) and `tables.tex` (booktabs).

In [1]:
import io
import itertools
import json
import os
import subprocess

import geopandas as gpd
import numpy as np
import pandas as pd
from verus.data import TimeWindowGenerator

city = "Lisbon"
article_commit = "7cef902"
out_dir = "../docs/thesis"
os.makedirs(out_dir, exist_ok=True)

tw = pd.read_csv("../data/time_windows/time_windows_T1g.csv")
tw_article = pd.read_csv("../data/time_windows/default_time_windows.csv")
scenarios = pd.read_csv("../data/time_windows/scenarios_T1g.csv")
scenarios["epoch"] = scenarios["evaluation_time"].map(TimeWindowGenerator.to_unix_epoch)
poti = pd.read_csv(f"../data/poti/{city.lower()}_dataset_buffered.csv")
with open(f"../data/vulnerability_layer/{city.lower()}/vulnerability_metadata.json") as f:
    vuln_meta = json.load(f)
cri = gpd.read_file(f"../data/multi_layers/{city}_multi_layer_all_scenarios_with_CRI.geojson").set_index("hex_id")
weights = pd.read_csv(f"../data/multi_layers/{city}_cri_weights.csv").set_index("scenario")

sc_ids = scenarios["scenario"].tolist()

## Helpers

In [2]:
def vi_at(table, category, epoch):
    act = table[(table["category"] == category) & (table["ts"] <= epoch) & (table["te"] >= epoch)]
    return sorted(set(act["vi"]))


def window_rows(table, reference):
    # One row per (category, day type, window, vi), compared with the reference table
    t = table.copy()
    start = pd.to_datetime(t["ts"], unit="s")
    end = pd.to_datetime(t["te"] + 1, unit="s")
    t["day_type"] = np.where(start.dt.dayofweek < 5, "Weekdays", "Weekends")
    t["start"] = start.dt.strftime("%H:%M")
    t["end"] = end.dt.strftime("%H:%M").replace("00:00", "24:00")
    rows = []
    for (cat, day_type, s, e, vi), g in t.groupby(["category", "day_type", "start", "end", "vi"]):
        first = g.sort_values("ts").iloc[0]
        # Probe the reference table every 15 min inside the window (first day of the group)
        probes = range(int(first["ts"]) + 450, int(first["te"]), 900)
        same = all(vi_at(reference, cat, p) == [vi] for p in probes)
        rows.append({"category": cat, "day_type": day_type, "start": s, "end": e, "vi": vi,
                     "status": "as in article" if same else "new or changed"})
    return pd.DataFrame(rows)


def top_set(series, frac=0.10):
    return set(series.nlargest(int(round(len(series) * frac))).index)


def pairwise(frame, cols, names):
    rows = []
    for a, b in itertools.combinations(cols, 2):
        ta, tb = top_set(frame[a]), top_set(frame[b])
        rows.append({"pair": f"{names[a]} / {names[b]}",
                     "spearman": frame[a].corr(frame[b], method="spearman"),
                     "top10_overlap": len(ta & tb) / len(ta | tb)})
    return pd.DataFrame(rows)


def latex_text(value):
    return str(value).replace("_", " ").replace("%", r"\%").replace("&", r"\&")


def to_latex(df, headers, caption, label, digits=3):
    d = df.copy()
    align = "".join("r" if pd.api.types.is_numeric_dtype(d[c]) else "l" for c in d.columns)
    for c in d.columns:
        if not pd.api.types.is_numeric_dtype(d[c]):
            d[c] = d[c].map(latex_text)
    d.columns = headers
    body = d.to_latex(index=False, escape=False, column_format=align,
                      float_format=lambda x: f"{x:.{digits}f}")
    tex = ("\\begin{table}[htbp]\n\\centering\n\\small\n"
           f"\\caption{{{caption}}}\n\\label{{{label}}}\n{body}\\end{{table}}\n\n")
    with open(tex_path, "a") as f:
        f.write(tex)


tex_path = os.path.join(out_dir, "tables.tex")
with open(tex_path, "w") as f:
    f.write("% Generated by notebook/10-thesis_tables.ipynb. Requires \\usepackage{booktabs}.\n\n")

## Table 1. Time-window table T1g

In [3]:
t1 = window_rows(tw, tw_article).sort_values(["category", "day_type", "start"])
t1.to_csv(os.path.join(out_dir, "table1_time_windows_T1g.csv"), index=False)
to_latex(t1, ["Category", "Days", "Start", "End", "$v_i$", "Status"],
         "Time windows of table T1g and the vulnerability index $v_i$ of each POTI category (window end "
         "exclusive). The status compares each window with the table used in the article.",
         "tab:time-windows-t1g", digits=1)
t1

,category,day_type,start,end,vi,status
0,attraction,Weekdays,09:00,17:00,0.5,as in article
1,attraction,Weekends,09:00,17:00,0.7,as in article
2,bus_station,Weekdays,07:00,09:00,0.5,as in article
3,bus_station,Weekdays,09:00,17:00,0.2,new or changed
4,bus_station,Weekdays,17:00,19:00,0.5,as in article
5,bus_station,Weekdays,19:00,22:00,0.2,new or changed
6,hospital,Weekdays,00:00,07:00,0.2,new or changed
7,hospital,Weekdays,07:00,10:00,0.8,as in article
8,hospital,Weekdays,10:00,16:00,0.4,as in article
9,hospital,Weekdays,16:00,19:00,0.8,as in article


## Table 2. Scenarios

In [4]:
rows = []
for _, sc in scenarios.iterrows():
    act = tw[(tw["ts"] <= sc["epoch"]) & (tw["te"] >= sc["epoch"])]
    vi = dict(zip(act["category"], act["vi"]))
    rows.append({"scenario": sc["scenario"].upper(), "regime": sc["regime"],
                 "evaluation_time": pd.Timestamp(sc["evaluation_time"]).strftime("%a %H:%M"),
                 "active_categories": ", ".join(f"{c} ({v:.1f})" for c, v in sorted(vi.items())),
                 "active_potis": int(poti["category"].isin(vi).sum())})
t2 = pd.DataFrame(rows)
t2.to_csv(os.path.join(out_dir, "table2_scenarios.csv"), index=False)
to_latex(t2, ["Scenario", "Regime", "Time", "Active categories ($v_i$)", "Active POTIs"],
         f"Scenarios, their evaluation times, the active POTI categories with $v_i$, and the number of "
         f"active POTIs (of {len(poti)}).", "tab:scenarios")
t2

,scenario,regime,evaluation_time,active_categories,active_potis
0,S1,Weekday morning rush,Mon 08:30,"bus_station (0.5), hospital (0.8), industrial ...",571
1,S2,Weekday late morning,Mon 11:00,"attraction (0.5), bus_station (0.2), hospital ...",385
2,S3,Weekday night,Mon 21:00,"bus_station (0.2), hospital (0.2), station (0.3)",124
3,S4,Saturday daytime,Sat 11:00,"attraction (0.7), hospital (0.2), mall (0.2), ...",325


## Table 3. Vulnerability and CRI per scenario

In [5]:
rows = []
for s in sc_ids:
    v, c = cri[f"vulnerability_{s}_value"], cri[f"{s}_cri"]
    w = weights.loc[s]
    rows.append({"scenario": s.upper(), "vuln_mean": v.mean(), "vuln_p95": v.quantile(0.95),
                 "w_vulnerability": w["w_vulnerability"], "w_response": w["w_response"], "w_risk": w["w_risk"],
                 "w_gini": w["w_gini"], "w_income": w["w_income"],
                 "cri_mean": c.mean(), "cri_iqr": c.quantile(0.75) - c.quantile(0.25)})
t3 = pd.DataFrame(rows)
t3.to_csv(os.path.join(out_dir, "table3_vulnerability_cri.csv"), index=False)
to_latex(t3, ["Scenario", r"$\bar{V}$", "$V_{95}$", "$w_V$", "$w_R$", "$w_F$", "$w_G$", "$w_I$",
              r"$\overline{\mathrm{CRI}}$", "IQR"],
         "Vulnerability $V$ per scenario (mean and 95th percentile, normalized by the maximum over the four "
         f"scenarios, {vuln_meta['max_vulnerability']:.3e}), entropy weights of the core indicators "
         "(vulnerability $w_V$, response $w_R$, flood risk $w_F$) and equity indicators (Gini $w_G$, "
         r"income $w_I$), and CRI mean and interquartile range ($\gamma = 0.5$).", "tab:vulnerability-cri")
t3

,scenario,vuln_mean,vuln_p95,w_vulnerability,w_response,w_risk,w_gini,w_income,cri_mean,cri_iqr
0,S1,0.551172,0.826562,0.157683,0.471869,0.370448,0.439039,0.560961,0.407122,0.207857
1,S2,0.360227,0.593079,0.243839,0.423604,0.332556,0.439039,0.560961,0.405159,0.213786
2,S3,0.119216,0.245734,0.379083,0.347840,0.273077,0.439039,0.560961,0.536416,0.269808
3,S4,0.375357,0.768785,0.343821,0.367594,0.288585,0.439039,0.560961,0.420354,0.239217


## Table 4. Differences between scenarios

In [6]:
cols = [f"{s}_cri" for s in sc_ids]
t4 = pairwise(cri, cols, {f"{s}_cri": s.upper() for s in sc_ids})
t4.to_csv(os.path.join(out_dir, "table4_scenario_pairs.csv"), index=False)
to_latex(t4, ["Pair", r"Spearman $\rho$", r"Top-10\% overlap"],
         r"CRI agreement between scenario pairs: Spearman correlation over the hexagons and Jaccard overlap "
         r"of the top 10\% priority hexagons.", "tab:scenario-pairs")
t4

,pair,spearman,top10_overlap
0,S1 / S2,0.907610,0.733509
1,S1 / S3,0.736795,0.549528
2,S1 / S4,0.758120,0.626238
3,S2 / S3,0.777422,0.560570
4,S2 / S4,0.879321,0.809917
5,S3 / S4,0.684521,0.506881


## Table 5. Article scenarios and revised scenarios

In [7]:
raw = subprocess.run(
    ["git", "show", f"{article_commit}:data/multi_layers/{city}_multi_layer_all_scenarios_with_CRI.geojson"],
    capture_output=True, check=True).stdout
article = gpd.read_file(io.BytesIO(raw)).set_index("hex_id")
art_cols = [f"s{i}_cri" for i in range(1, 5)]
art_names = {"s1_cri": "Sat 10:20", "s2_cri": "Mon 08:40", "s3_cri": "Mon 12:30", "s4_cri": "Mon 17:30"}
rev_names = {f"{s}_cri": pd.Timestamp(e).strftime("%a %H:%M") for s, e in zip(sc_ids, scenarios["evaluation_time"])}


def summary(frame, cols, names, label):
    p = pairwise(frame, cols, names)
    worst = p.loc[p["spearman"].idxmax()]
    return {"scenario_set": label, "mean_spearman": p["spearman"].mean(), "max_spearman": p["spearman"].max(),
            "most_similar_pair": worst["pair"],
            "mean_top10_overlap": p["top10_overlap"].mean(), "max_top10_overlap": p["top10_overlap"].max()}


t5 = pd.DataFrame([summary(article, art_cols, art_names, "Article (default windows)"),
                   summary(cri, cols, rev_names, "Revised (T1g regimes)")])
t5.to_csv(os.path.join(out_dir, "table5_article_vs_revised.csv"), index=False)
to_latex(t5, ["Scenario set", r"Mean $\rho$", r"Max $\rho$", "Most similar pair", "Mean overlap", "Max overlap"],
         r"Separation between scenarios in the article and in the revised set, over all scenario pairs. "
         r"Lower Spearman $\rho$ and lower top-10\% overlap mean more distinct scenarios.",
         "tab:article-vs-revised")
t5

,scenario_set,mean_spearman,max_spearman,most_similar_pair,mean_top10_overlap,max_top10_overlap
0,Article (default windows),0.889140,0.99487,Mon 08:40 / Mon 17:30,0.709875,0.955357
1,Revised (T1g regimes),0.790632,0.90761,Mon 08:30 / Mon 11:00,0.631107,0.809917


In [8]:
print(open(tex_path).read())

% Generated by notebook/10-thesis_tables.ipynb. Requires \usepackage{booktabs}.

\begin{table}[htbp]
\centering
\small
\caption{Time windows of table T1g and the vulnerability index $v_i$ of each POTI category (window end exclusive). The status compares each window with the table used in the article.}
\label{tab:time-windows-t1g}
\begin{tabular}{llllrl}
\toprule
Category & Days & Start & End & $v_i$ & Status \\
\midrule
attraction & Weekdays & 09:00 & 17:00 & 0.5 & as in article \\
attraction & Weekends & 09:00 & 17:00 & 0.7 & as in article \\
bus station & Weekdays & 07:00 & 09:00 & 0.5 & as in article \\
bus station & Weekdays & 09:00 & 17:00 & 0.2 & new or changed \\
bus station & Weekdays & 17:00 & 19:00 & 0.5 & as in article \\
bus station & Weekdays & 19:00 & 22:00 & 0.2 & new or changed \\
hospital & Weekdays & 00:00 & 07:00 & 0.2 & new or changed \\
hospital & Weekdays & 07:00 & 10:00 & 0.8 & as in article \\
hospital & Weekdays & 10:00 & 16:00 & 0.4 & as in article \\
hospital